# 16S Analyses of the Longitudinal Acne Study
## CTF and RPCA Plots

Date created: 10/18/2024

Notebook author: Britta De Pessemier 

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

This notebook plots the following:
- CTF plots 
- RPCA plots

In [83]:
# Import essential Python packages
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
from adjustText import adjust_text

# Scientific and statistical packages
import scipy.stats as ss
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults
from skbio.stats.distance import permanova

# BIOM and Qiime2 related packages
import biom
from biom import load_table
from qiime2 import Artifact

# Gemelli RPCA and rarefaction utilities
from gemelli.rpca import rpca # Optional: import auto_rpca if you use auto_rpca later
from gemelli.utils import qc_rarefaction


In [84]:
# Function to draw an ellipse based on the covariance of the points
def draw_ellipse(mean, cov, ax, color, label=None, scale_factor=1.5):
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    
    # Calculate the angle between the x-axis and the largest eigenvector
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    
    # Width and height of the ellipse (scaling applied)
    width, height = 2 * np.sqrt(eigenvalues) * scale_factor
    
    # Create the ellipse and add it to the plot
    ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
    ax.add_patch(ell)

## CTF 
Analysis per participant skin site location C1/C2 for healthy and C1/C2/C3 for acne 


In [85]:

# Define the color palette for the groups
group_colors = {
    "Healthy": "#58a35b",  # Color for Healthy
    "Acne_L": "#dd7966",   # Color for Acne Lesional
    "Acne_NL": "#6ab2bd"   # Color for Acne Non-lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/distance_matrix.qza')
    
    # View as a skbio DistanceMatrix object
    distance_matrix = table.view(DistanceMatrix)
    
    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
    # Display the DataFrame
    print(data)
    
    # Load the subject_biplot artifact
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/subject_biplot.qza')
    
    # View as an OrdinationResults object
    ordination = biplot.view(OrdinationResults)
    
    # Extract the sample coordinates as a DataFrame
    sample_coordinates = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    
    # Extract the feature (or variable) coordinates as a DataFrame
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    
    # Display the DataFrames
    print("Sample Coordinates:")
    print(sample_coordinates)
    print("\nFeature Coordinates:")
    print(feature_coordinates)
    
    # Extract the proportion explained
    proportion_explained = ordination.proportion_explained
    
    # Convert to percentages for PC1 and PC2
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    print(f"PC1 explains {pc1_explained_variance:.2f}% of the variance")
    print(f"PC2 explains {pc2_explained_variance:.2f}% of the variance")
    
    # Rename the sample coordinates DataFrame to spca_df
    spca_df = sample_coordinates
    
    # Load the metadata from the subject-metadata.tsv file
    metadata_df = pd.read_csv(f'../Data/16S/CTF/ctf-results_{folder}/subject-metadata.tsv', sep='\t')
    
    # Map the index of spca_df (sample names) to the SampleID in metadata_df to get the 'group'
    spca_df["Group"] = spca_df.index.map(metadata_df.set_index("#SampleID")["group"])
    
    # Rename the columns of spca_df for clarity (PC1, PC2, PC3)
    spca_df.columns = ['PC1', 'PC2', 'PC3', 'Group']
    
    # Display the updated spca_df
    print(spca_df)
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],  # Semi-transparent fill color
            edgecolor=group_colors[group_name],  # Colored border
            alpha=0.8,  # Transparency for the fill
            linewidth=0.5,  # Thickness of the borders
            ax=ax,
            label=group_name_mapping[group_name]
        )
    
        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)
    
        # Draw the ellipse for the group (with scale factor applied)
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)
    
        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
    
    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
   # Add the title consistently in the top-left corner using axes coordinates
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

    # Customize legend
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{file_suffix}.png", dpi=600)
    plt.show()


Processing folder: 174950
                   LAMI.RD017.D14.C1  LAMI.RD003.D28.C2  LAMI.RD011.D0.C1  \
LAMI.RD017.D14.C1           0.000000          15.187849         13.257771   
LAMI.RD003.D28.C2          15.187849           0.000000          5.475794   
LAMI.RD011.D0.C1           13.257771           5.475794          0.000000   
LAMI.RD011.D28.C1          13.083151           5.925958          0.674414   
LAMI.RD006.D0.C2           13.581639           6.303939          0.955279   
...                              ...                ...               ...   
LAMI.RD310.D9.C1           10.333933           6.415692          5.613001   
LAMI.RD310.D21.C1          10.123376           6.583676          5.914402   
LAMI.RD310.D0.C1           10.588279           6.113444          5.464744   
LAMI.RD310.D14.C1          10.385060           6.272099          5.531075   
LAMI.RD310.D28.C1          10.042077           6.342499          5.498209   

                   LAMI.RD011.D28.C1  LAMI.RD006.

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2523675088.py:107: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 

Processing folder: 174951
                   LAMI.RD011.D14.C1  LAMI.RD006.D14.C2  LAMI.RD006.D14.C1  \
LAMI.RD011.D14.C1           0.000000           6.479502           7.139171   
LAMI.RD006.D14.C2           6.479502           0.000000           0.825376   
LAMI.RD006.D14.C1           7.139171           0.825376           0.000000   
LAMI.RD018.D0.C1            6.718586           5.770834           6.472288   
LAMI.RD018.D0.C2            6.491061           5.198449           5.919783   
...                              ...                ...                ...   
LAMI.RD314.D16.C1           5.805068           7.323252           7.994251   
LAMI.RD314.D21.C1           5.700178           7.377919           8.065342   
LAMI.RD314.D0.C1            6.335841           7.256653           7.877528   
LAMI.RD314.D28.C1           5.713597           7.539581           8.218511   
LAMI.RD314.D14.C1           5.861592           7.324871           7.996571   

                   LAMI.RD018.D0.C1  

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2523675088.py:138: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## CTF colored by severity group

In [87]:
# Create scatter plot with different colors based on the Group column
for group_name, group in spca_df.groupby('Group'):
    # Calculate mean and covariance matrix for the group
    points = group[["PC1", "PC2"]].values

    # Debugging: Check the shape of points
    print(f"Points shape for group '{group_name}': {points.shape}")

    if points.shape[0] < 2:  # Ensure at least 2 points for covariance
        print(f"Not enough points to compute covariance for group '{group_name}'")
        continue  # Correctly skip this group if not enough points

    # Calculate mean and covariance matrix for the group
    mean = np.mean(points, axis=0)
    cov = np.cov(points, rowvar=False)

    # Draw the ellipse for the group (with scale factor applied)
    draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

    # Add a subtle star (8-pointed) at the mean location for each group
    ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)


Points shape for group 'absent Acne_NL': (9, 2)
Points shape for group 'absent Healthy': (14, 2)
Points shape for group 'high Acne_L': (10, 2)
Points shape for group 'low Acne_L': (18, 2)
Points shape for group 'low Acne_NL': (9, 2)
Points shape for group 'moderate Acne_L': (23, 2)


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1708207561.py:21: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3

In [88]:
# Define the color palette for the groups
group_colors = {
    "absent Healthy": "#006f00",
    "absent Acne_NL": "#9af09b",
    "low Acne_NL": "#b6ddea", 
    "low Acne_L": "#ff676c",
    "moderate Acne_L": "#ca0000", 
    "high Acne_L": "#960000"
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "absent Healthy": "absent Healthy",
    "absent Acne_NL": "absent Acne non-lesional",
    "low Acne_NL": "low Acne non-lesional", 
    "low Acne_L": "low Acne lesional",
    "moderate Acne_L": "moderate Acne lesional", 
    "high Acne_L": "high Acne lesional"
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/distance_matrix.qza')
    
    # View as a skbio DistanceMatrix object
    distance_matrix = table.view(DistanceMatrix)
    
    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
    # Display the DataFrame
    print(data)
    
    # Load the subject_biplot artifact
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/subject_biplot.qza')
    
    # View as an OrdinationResults object
    ordination = biplot.view(OrdinationResults)
    
    # Extract the sample coordinates as a DataFrame
    sample_coordinates = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    
    # Extract the feature (or variable) coordinates as a DataFrame
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    
    # Display the DataFrames
    print("Sample Coordinates:")
    print(sample_coordinates)
    print("\nFeature Coordinates:")
    print(feature_coordinates)
    
    # Extract the proportion explained
    proportion_explained = ordination.proportion_explained
    
    # Convert to percentages for PC1 and PC2
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    print(f"PC1 explains {pc1_explained_variance:.2f}% of the variance")
    print(f"PC2 explains {pc2_explained_variance:.2f}% of the variance")
    
    # Rename the sample coordinates DataFrame to spca_df
    spca_df = sample_coordinates
    
    # Load the metadata from the subject-metadata.tsv file
    metadata_df = pd.read_csv(f'../Data/16S/CTF/ctf-results_{folder}/subject-metadata_severity_group.tsv', sep='\t')
    
    # Map the index of spca_df (sample names) to the SampleID in metadata_df to get the 'group'
    spca_df["Group"] = spca_df.index.map(metadata_df.set_index("#SampleID")["severity_group"])
    
    # Rename the columns of spca_df for clarity (PC1, PC2, PC3)
    spca_df.columns = ['PC1', 'PC2', 'PC3', 'Group']
    
    # Display the updated spca_df
    print(spca_df)
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('Group'):
        # Calculate points for the group
        points = group[["PC1", "PC2"]].values
        
        # Debugging: Check the shape of points
        print(f"Points shape for group '{group_name}': {points.shape}")

        # Check if the group has enough points for covariance calculation
        if points.shape[0] < 2:
            print(f"Not enough points to compute covariance for group '{group_name}'. Skipping this group.")
            continue  # Skip this group if not enough points

        # Create scatter plot for the group
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],  # Semi-transparent fill color
            edgecolor=group_colors[group_name],  # Colored border
            alpha=0.8,  # Transparency for the fill
            linewidth=0.5,  # Thickness of the borders
            ax=ax,
            label=group_name_mapping[group_name]
        )

        # Calculate mean and covariance matrix for the group
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)

        # Draw the ellipse for the group (with scale factor applied)
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title consistently in the top-left corner using axes coordinates
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

    # Customize legend
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{file_suffix}_severity_group.png", dpi=600)
    plt.show()


Processing folder: 174950
                   LAMI.RD017.D14.C1  LAMI.RD003.D28.C2  LAMI.RD011.D0.C1  \
LAMI.RD017.D14.C1           0.000000          15.187849         13.257771   
LAMI.RD003.D28.C2          15.187849           0.000000          5.475794   
LAMI.RD011.D0.C1           13.257771           5.475794          0.000000   
LAMI.RD011.D28.C1          13.083151           5.925958          0.674414   
LAMI.RD006.D0.C2           13.581639           6.303939          0.955279   
...                              ...                ...               ...   
LAMI.RD310.D9.C1           10.333933           6.415692          5.613001   
LAMI.RD310.D21.C1          10.123376           6.583676          5.914402   
LAMI.RD310.D0.C1           10.588279           6.113444          5.464744   
LAMI.RD310.D14.C1          10.385060           6.272099          5.531075   
LAMI.RD310.D28.C1          10.042077           6.342499          5.498209   

                   LAMI.RD011.D28.C1  LAMI.RD006.

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/370074417.py:124: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3

Points shape for group 'moderate Acne_L': (11, 2)


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/370074417.py:155: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/370074417.py:124: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)


Processing folder: 174951
                   LAMI.RD011.D14.C1  LAMI.RD006.D14.C2  LAMI.RD006.D14.C1  \
LAMI.RD011.D14.C1           0.000000           6.479502           7.139171   
LAMI.RD006.D14.C2           6.479502           0.000000           0.825376   
LAMI.RD006.D14.C1           7.139171           0.825376           0.000000   
LAMI.RD018.D0.C1            6.718586           5.770834           6.472288   
LAMI.RD018.D0.C2            6.491061           5.198449           5.919783   
...                              ...                ...                ...   
LAMI.RD314.D16.C1           5.805068           7.323252           7.994251   
LAMI.RD314.D21.C1           5.700178           7.377919           8.065342   
LAMI.RD314.D0.C1            6.335841           7.256653           7.877528   
LAMI.RD314.D28.C1           5.713597           7.539581           8.218511   
LAMI.RD314.D14.C1           5.861592           7.324871           7.996571   

                   LAMI.RD018.D0.C1  

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/370074417.py:124: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3

Points shape for group 'high Acne_L': (1, 2)
Not enough points to compute covariance for group 'high Acne_L'. Skipping this group.
Points shape for group 'low Acne_L': (8, 2)
Points shape for group 'low Acne_NL': (3, 2)
Points shape for group 'moderate Acne_L': (11, 2)


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/370074417.py:155: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


# Rerun CTF for the severity groups 


In [89]:

# Define the color palette for the groups
group_colors = {
    "absent Healthy": "#006f00",
    "absent Acne_NL": "#9af09b",
    "low Acne_NL": "#b6ddea", 
    "low Acne_L": "#ff676c",
    "moderate Acne_L": "#ca0000", 
    "high Acne_L": "#960000"
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "absent Healthy": "absent Healthy",
    "absent Acne_NL": "absent Acne non-lesional",
    "low Acne_NL": "low Acne non-lesional", 
    "low Acne_L": "low Acne lesional",
    "moderate Acne_L": "moderate Acne lesional", 
    "high Acne_L": "high Acne lesional"
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_subject_ID_CC_LLS_{folder}/distance_matrix.qza')
    
    # View as a skbio DistanceMatrix object
    distance_matrix = table.view(DistanceMatrix)
    
    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
    # Display the DataFrame
    print(data)
    
    # Load the subject_biplot artifact
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_subject_ID_CC_LLS_{folder}/subject_biplot.qza')
    
    # View as an OrdinationResults object
    ordination = biplot.view(OrdinationResults)
    
    # Extract the sample coordinates as a DataFrame
    sample_coordinates = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    
    # Extract the feature (or variable) coordinates as a DataFrame
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    
    # Display the DataFrames
    print("Sample Coordinates:")
    print(sample_coordinates)
    print("\nFeature Coordinates:")
    print(feature_coordinates)
    
    # Extract the proportion explained
    proportion_explained = ordination.proportion_explained
    
    # Convert to percentages for PC1 and PC2
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    print(f"PC1 explains {pc1_explained_variance:.2f}% of the variance")
    print(f"PC2 explains {pc2_explained_variance:.2f}% of the variance")
    
    # Rename the sample coordinates DataFrame to spca_df
    spca_df = sample_coordinates
    
    # Load the metadata from the subject-metadata.tsv file
    metadata_df = pd.read_csv(f'../Data/16S/CTF/ctf-results_subject_ID_CC_LLS_{folder}/subject-metadata_subject_ID_CC_LLS.tsv', sep='\t')
    
    # Map the index of spca_df (sample names) to the SampleID in metadata_df to get the 'group'
    spca_df["Group"] = spca_df.index.map(metadata_df.set_index("#SampleID")["severity_group"])
    
    # Rename the columns of spca_df for clarity (PC1, PC2, PC3)
    spca_df.columns = ['PC1', 'PC2', 'PC3', 'Group']
    
    # Display the updated spca_df
    print(spca_df)
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('Group'):
        # Calculate points for the group
        points = group[["PC1", "PC2"]].values
        
        # Debugging: Check the shape of points
        print(f"Points shape for group '{group_name}': {points.shape}")

        # Check if the group has enough points for covariance calculation
        if points.shape[0] < 2:
            print(f"Not enough points to compute covariance for group '{group_name}'. Skipping this group.")
            continue  # Skip this group if not enough points

        # Create scatter plot for the group
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],  # Semi-transparent fill color
            edgecolor=group_colors[group_name],  # Colored border
            alpha=0.8,  # Transparency for the fill
            linewidth=0.5,  # Thickness of the borders
            ax=ax,
            label=group_name_mapping[group_name]
        )

        # Calculate mean and covariance matrix for the group
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)

        # Draw the ellipse for the group (with scale factor applied)
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title consistently in the top-left corner using axes coordinates
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

    # Customize legend
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{file_suffix}_subject_ID_CC_LLS_severity_group.png", dpi=600)
    plt.show()


Processing folder: 174950
                   LAMI.RD304.D11.C2  LAMI.RD304.D28.C3  LAMI.RD305.D23.C2  \
LAMI.RD304.D11.C2           0.000000          12.477221           1.905424   
LAMI.RD304.D28.C3          12.477221           0.000000          12.676795   
LAMI.RD305.D23.C2           1.905424          12.676795           0.000000   
LAMI.RD317.D25.C1           7.319722           7.781132           7.031533   
LAMI.RD305.D28.C2          11.895487           5.032716          11.736082   
...                              ...                ...                ...   
LAMI.RD313.D9.C1            5.847184           6.974892           5.878797   
LAMI.RD313.D16.C1           5.985790           7.256817           5.794207   
LAMI.RD313.D0.C1            5.071833           7.522883           5.205771   
LAMI.RD313.D14.C1           5.076282           7.552796           5.191125   
LAMI.RD313.D28.C1           4.899011           7.841812           4.897679   

                   LAMI.RD317.D25.C1 

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2987052189.py:124: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 

Points shape for group 'low Acne_NL': (9, 2)
Points shape for group 'moderate Acne_L': (23, 2)


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2987052189.py:155: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2987052189.py:124: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

Processing folder: 174951
                   LAMI.RD011.D14.C1  LAMI.RD308.D9.C3  LAMI.RD310.D18.C1  \
LAMI.RD011.D14.C1           0.000000         14.545560          10.672837   
LAMI.RD308.D9.C3           14.545560          0.000000           4.442995   
LAMI.RD310.D18.C1          10.672837          4.442995           0.000000   
LAMI.RD306.D28.C3          10.782609          5.333329           4.713322   
LAMI.RD306.D14.C1          15.085672          2.714641           5.585196   
...                              ...               ...                ...   
LAMI.RD313.D9.C1            1.718107         13.169930           9.183365   
LAMI.RD313.D16.C1           6.426703          8.352727           4.253405   
LAMI.RD313.D0.C1            3.568725         11.199180           7.227267   
LAMI.RD313.D28.C1           2.906889         11.769855           7.784092   
LAMI.RD313.D14.C1           4.026790         10.625794           6.672868   

                   LAMI.RD306.D28.C3  LAMI.RD306.

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2987052189.py:124: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 

Points shape for group 'low Acne_NL': (9, 2)
Points shape for group 'moderate Acne_L': (23, 2)


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2987052189.py:155: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## RPCA

In [90]:
# import Python packages
import pandas as pd
import numpy as np
from numpy import array
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
import scipy
import scipy.stats as ss
from skbio.stats.distance import permanova
import biom
from biom import load_table
from gemelli.rpca import rpca #import auto_rpca
from gemelli.utils import qc_rarefaction
from skbio.stats.distance import permanova, DistanceMatrix
from adjustText import adjust_text

In [91]:
# Set the random seed for reproducibility
np.random.seed(123)
# Define the color palette for the groups
group_colors = {
    "Healthy": "#58a35b",  # Color for Healthy
    "Acne_L": "#dd7966",   # Color for Acne Lesional
    "Acne_NL": "#6ab2bd"   # Color for Acne Non-lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom'),
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom')
]

# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table and perform RPCA
    biom_tbl = biom.load_table(biom_file)
    
    # Perform RPCA with auto rank estimation
    np.seterr(divide='ignore')
    ordination, distance = rpca(biom_tbl)

    # Read the metadata file
    metadata_file = '../Metadata/metadata_final_18062024.tsv'
    metadata_df = pd.read_csv(metadata_file, sep='\t')

    # Extract and view sample ordinations from RPCA result
    spca_df = pd.DataFrame(ordination.samples)
    spca_df["Group"] = spca_df.index.map(metadata_df.set_index("SampleID")["group"])
    
    # calculate permanova F-statistic
    pnova_res = permanova(distance, spca_df, "Group")
    p_value = pnova_res['p-value']
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],  # Semi-transparent fill color
            edgecolor=group_colors[group_name],  # Colored border
            alpha=0.8,  # Transparency for the fill
            linewidth=0.5,  # Thickness of the borders
            ax=ax,
            label=group_name_mapping[group_name]
        )
    
        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)
    
        # Draw the ellipse for the group (with scale factor applied)
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)
    
        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
    
    # Add axis labels and include the explained variance percentages
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title based on the region being analyzed
    ax.text(0.05, 1.10, f'RPCA - 16S rRNA ({region_title})', fontsize=8, va='top', ha='left', transform=ax.transAxes)
    
    # Add the p-value from PERMANOVA
    ax.text(-0.18, 0.27, 'PERMANOVA', fontsize=7)
    ax.text(-0.18, 0.25, f'***p-val = {p_value:.3f}', fontsize=7)

    # Customize legend in the top-right corner
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7,
        loc='upper right'
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{region_title}_RPCA.png", dpi=600)
    plt.show()


Processing dataset: 174950 - V1-V3


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1111988175.py:73: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3

Processing dataset: 174951 - V4


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1111988175.py:73: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3

In [92]:
#look at the severity
# Set the random seed for reproducibility
np.random.seed(123)
# Define the color palette for the groups
local_lesion_severity_colors = {
    "0": "#006f00",
    "1": "#9af09b", # Color for Healthy
    "2": "#b6ddea",   # Color for Acne Lesional
    "3": "#a4a4ff",   # Color for Acne Non-lesional
    "4": "#000096",
    "5": "#ff676c",
    "6": "#960000"
}

# Map the old group names to new ones for the legend
group_name_mapping  = {
    "0":"0",
    "1": "1",  
    "2": "2",   
    "3": "3",   
    "4": "4",
    "5": "5",
    "6": "6"
}

# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom'),
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom')

]

# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table and perform RPCA
    biom_tbl = biom.load_table(biom_file)
    
    # Perform RPCA with auto rank estimation
    np.seterr(divide='ignore')
    ordination, distance = rpca(biom_tbl)

    # Read the metadata file
    metadata_file = '../Metadata/metadata_final_18062024.tsv'
    metadata_df = pd.read_csv(metadata_file, sep='\t')
    # Ensure local_lesion_severity is treated as a string (categorical variable)
    metadata_df["local_lesion_severity"] = metadata_df["local_lesion_severity"].astype(str)

    # Clear the figure to avoid overlapping plots
    plt.clf()
    # Extract and view sample ordinations from RPCA result
    spca_df = pd.DataFrame(ordination.samples)
    spca_df["local_lesion_severity"] = spca_df.index.map(metadata_df.set_index("SampleID")["local_lesion_severity"])
    
    # calculate permanova F-statistic
    pnova_res = permanova(distance, spca_df, "local_lesion_severity")
    p_value = pnova_res['p-value']
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('local_lesion_severity'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=local_lesion_severity_colors[group_name],  # Semi-transparent fill color
            edgecolor=local_lesion_severity_colors[group_name],  # Colored border
            alpha=0.8,  # Transparency for the fill
            linewidth=0.5,  # Thickness of the borders
            ax=ax,
            label=group_name_mapping[group_name]
        )
    
# Calculate mean and covariance matrix for the group if more than 2 points
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        
        if len(group) > 2:

            cov = np.cov(points, rowvar=False)
    
            # Draw the ellipse for the group (with scale factor applied)
            draw_ellipse(mean, cov, ax, local_lesion_severity_colors[group_name], scale_factor=2)
    
        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=local_lesion_severity_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
    # Add axis labels and include the explained variance percentages
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title based on the region being analyzed
    ax.text(0.05, 1.10, f'RPCA - 16S rRNA ({region_title})', fontsize=8, va='top', ha='left', transform=ax.transAxes)
    
    # Add the p-value from PERMANOVA
    ax.text(-0.18, 0.27, 'PERMANOVA', fontsize=7)
    ax.text(-0.18, 0.25, f'***p-val = {p_value:.3f}', fontsize=7)

    # Customize legend in the top-right corner
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7,
        loc='upper right'
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{region_title}_local_lesion_severity_RPCA.png", dpi=600)
    plt.show()


Processing dataset: 174951 - V4


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/937241870.py:90: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=local_lesion_severity_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated sin

Processing dataset: 174950 - V1-V3


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/937241870.py:90: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=local_lesion_severity_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated sin

In [122]:
# Read the metadata file
metadata_file = '../Metadata/metadata_final_18062024.tsv'
metadata_df = pd.read_csv(metadata_file, sep='\t')

# Specify the group column
group_column = 'group' 

# Calculate the count of local_lesion_severity for each group
severity_counts = (
    metadata_df.groupby([group_column, 'local_lesion_severity'])
    .size()
    .unstack(fill_value=0)
)

# Print the result
print(severity_counts)


local_lesion_severity   0   1   2   3   4   5  6
group                                           
Acne_L                  0  38  32  36  28  19  6
Acne_NL                27  18   5   0   0   0  0
Healthy                57   0   0   0   0   0  0


In [123]:
# Create a mapping for severity levels
severity_mapping = {
    0: 'absent',      # You can change 'none' to another term if desired
    1: 'low',
    2: 'low',
    3: 'moderate',
    4: 'moderate',
    5: 'high',
    6: 'high'
}

# Add the severity_level column to metadata_df
metadata_df['severity_level'] = metadata_df['local_lesion_severity'].map(severity_mapping)

# Print the updated metadata DataFrame
print(metadata_df)

              SampleID c_zone  \
0    LAMI.RD308.D16.C1     C1   
1    LAMI.RD310.D21.C1     C1   
2    LAMI.RD305.D21.C3     C3   
3    LAMI.RD306.D18.C2     C2   
4     LAMI.RD306.D7.C2     C2   
..                 ...    ...   
261  LAMI.RD317.D21.C1     C1   
262   LAMI.RD001.D0.C1     C1   
263  LAMI.RD014.D14.C2     C2   
264   LAMI.RD314.D0.C1     C1   
265  LAMI.RD001.D14.C2     C2   

    visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face  \
0                                       not applicable                  
1                                                   72                  
2                                                   69                  
3                                       not applicable                  
4                                                   90                  
..                                                 ...                  
261                                                 77                  
262                

In [124]:
# Create the combined column for severity and group
metadata_df['severity_group'] = metadata_df['severity_level'] + ' ' + metadata_df['group']


In [125]:
# Print unique values in the severity_group column
unique_severity_groups = metadata_df['severity_group'].unique()
print(unique_severity_groups)


['moderate Acne_L' 'absent Acne_NL' 'low Acne_L' 'absent Healthy'
 'low Acne_NL' 'high Acne_L']


In [54]:

# Set the random seed for reproducibility
np.random.seed(123)

# Define the color palette for the groups
severity_group_colors = {
    "absent Healthy": "#006f00",
    "absent Acne_NL":"#9af09b",
    "low Acne_NL": "#b6ddea", 
    "low Acne_L": "#ff676c",
    "moderate Acne_L": "#ca0000", 
    "high Acne_L": "#960000"
}

# Define the shape for each group
group_shape_mapping = {
    "Healthy": "o",    # Circle
    "Acne_NL": "^",    # Triangle
    "Acne_L": "D",     # Star
}

# Define the order for the legend
severity_order = ["absent Healthy", "absent Acne_NL", "low Acne_NL", "low Acne_L", "moderate Acne_L","high Acne_L"]

# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom'),
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom')
]

# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table and perform RPCA
    biom_tbl = biom.load_table(biom_file)
    
    # Perform RPCA with auto rank estimation
    np.seterr(divide='ignore')
    ordination, distance = rpca(biom_tbl)

    # Clear the figure to avoid overlapping plots
    plt.clf()
    # Extract and view sample ordinations from RPCA result
    spca_df = pd.DataFrame(ordination.samples)
    spca_df["severity_group"] = spca_df.index.map(metadata_df.set_index("SampleID")["severity_group"])
    spca_df["group"] = spca_df.index.map(metadata_df.set_index("SampleID")["group"])  # Add group mapping
    
    # Calculate permanova F-statistic
    pnova_res = permanova(distance, spca_df, "severity_group")
    p_value = pnova_res['p-value']
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors and shapes
    for severity_group in severity_order:
        group = spca_df[spca_df['severity_group'] == severity_group]
        for group_name, group_data in group.groupby('group'):
            sns.scatterplot(
                data=group_data,
                x="PC1",
                y="PC2",
                facecolor=severity_group_colors[severity_group],
                edgecolor=severity_group_colors[severity_group],
                alpha=0.8,
                linewidth=0.5,
                ax=ax,
                label=severity_group,
                marker=group_shape_mapping[group_name],  # Use shape mapping for each group
                s=10
            )
            
            # Calculate mean and covariance matrix for the group if more than 2 points
            points = group_data[["PC1", "PC2"]].values
            mean = np.mean(points, axis=0)
            
            if len(group_data) > 2:
                cov = np.cov(points, rowvar=False)
                draw_ellipse(mean, cov, ax, severity_group_colors[severity_group], scale_factor=2)

            # Add a subtle star (8-pointed) at the mean location for each group
            ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group], 
                       marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Add axis labels and include the explained variance percentages
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)

    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title based on the region being analyzed
    ax.text(0.05, 1.10, f'RPCA - 16S rRNA ({region_title})', fontsize=8, va='top', ha='left', transform=ax.transAxes)
    
    # Add the p-value from PERMANOVA
    ax.text(-0.18, 0.27, 'PERMANOVA', fontsize=7)
    ax.text(-0.18, 0.25, f'***p-val = {p_value:.3f}', fontsize=7)

    # Customize legend in the top-right corner
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7,
        loc='upper right'
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{region_title}_severity_group_RPCA.png", dpi=600)
    plt.show()


Processing dataset: 174951 - V4


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/847953302.py:83: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group],
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor rele

Processing dataset: 174950 - V1-V3


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/847953302.py:83: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group],
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor rele

In [ ]:
# Now filter out the samples that match 'low Acne_NL'
samples_to_drop = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']

# Get the number of samples that are being dropped
num_dropped_samples = len(samples_to_drop)

# Drop the samples
metadata_df = metadata_df[metadata_df['severity_group'] != 'low Acne_NL']

# Output the number of dropped samples
print(f'Number of samples dropped: {num_dropped_samples}')

In [121]:

# Set the random seed for reproducibility
np.random.seed(123)

# Define the color palette for the groups
severity_group_colors = {
    "absent Healthy": "#006f00",
    "absent Acne_NL":"#9af09b", 
    "low Acne_L": "#ff676c",
    "moderate Acne_L": "#ca0000", 
    "high Acne_L": "#960000"
}

# Define the shape for each group
group_shape_mapping = {
    "Healthy": "o",    # Circle
    "Acne_NL": "^",    # Triangle
    "Acne_L": "D",     # Star
}

# Define the order for the legend
severity_order = ["absent Healthy", "absent Acne_NL", "low Acne_NL", "low Acne_L", "moderate Acne_L","high Acne_L"]

# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom')#,
    #('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom')
]


# First, filter the metadata to get the samples to be removed
samples_to_remove = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']['SampleID'].tolist()

# Function to filter the biom table based on the samples to keep
def filter_biom_table(biom_table, samples_to_remove):
    # Get all sample IDs in the biom table
    all_samples = biom_table.ids(axis='sample')
    
    # Filter the samples, excluding the ones to be removed
    samples_to_keep = [sample for sample in all_samples if sample not in samples_to_remove]
    
    # Subset the biom table to only include the samples to keep
    filtered_biom_table = biom_table.filter(samples_to_keep, axis='sample', inplace=False)
    
    return filtered_biom_table, all_samples, samples_to_keep

# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table
    biom_tbl = biom.load_table(biom_file)
    
    # Filter the biom table to remove unwanted samples
    biom_tbl_filtered, original_samples, filtered_samples = filter_biom_table(biom_tbl, samples_to_remove)
    
    # Check the samples that were removed
    dropped_samples = set(original_samples) - set(filtered_samples)
    
    print(f"Number of samples in original table: {len(original_samples)}")
    print(f"Number of samples after filtering: {len(filtered_samples)}")
    print(f"Samples that were removed: {dropped_samples}")
    
    # Verify if the dropped samples match the expected samples
    if set(samples_to_remove) == dropped_samples:
        print(f"Correct samples were dropped: {len(dropped_samples)} samples removed.")
    else:
        print("There is a mismatch in the samples dropped. Please check!")




Processing dataset: 174951 - V4
Number of samples in original table: 217
Number of samples after filtering: 199
Samples that were removed: {'LAMI.RD306.D28.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD310.D7.C3', 'LAMI.RD304.D7.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD314.D14.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD314.D28.C3'}
There is a mismatch in the samples dropped. Please check!


In [108]:
# Verify if the samples to remove are present in the original dataset
samples_not_in_biom = set(samples_to_remove) - set(original_samples)

if samples_not_in_biom:
    print(f"The following samples to remove are not present in the BIOM table: {samples_not_in_biom}")
else:
    print("All samples to remove are present in the BIOM table.")
    
# Verify if the dropped samples match the expected samples
expected_set = set(samples_to_remove)
dropped_set = set(dropped_samples)

if expected_set == dropped_set:
    print(f"Correct samples were dropped: {len(dropped_samples)} samples removed.")
else:
    print("There is a mismatch in the samples dropped. Please check!")
    
    # Print the differences
    print("Samples expected to be removed but not found in the BIOM table:")
    print(expected_set - dropped_set)
    
    print("Samples dropped from the BIOM table but not expected to be removed:")
    print(dropped_set - expected_set)


The following samples to remove are not present in the BIOM table: {'LAMI.RD317.D21.C3', 'LAMI.RD308.D21.C3', 'LAMI.RD306.D7.C3', 'LAMI.RD319.D21.C3', 'LAMI.RD306.D0.C3'}
There is a mismatch in the samples dropped. Please check!
Samples expected to be removed but not found in the BIOM table:
{'LAMI.RD317.D21.C3', 'LAMI.RD308.D21.C3', 'LAMI.RD306.D7.C3', 'LAMI.RD319.D21.C3', 'LAMI.RD306.D0.C3'}
Samples dropped from the BIOM table but not expected to be removed:
set()


In [127]:
# Set the random seed for reproducibility
np.random.seed(123)

# Define the color palette for the groups
severity_group_colors = {
    "absent Healthy": "#006f00",
    "absent Acne_NL": "#9af09b",
    "low Acne_L": "#ff676c",
    "moderate Acne_L": "#ca0000", 
    "high Acne_L": "#960000"
}

# Define the shape for each group
group_shape_mapping = {
    "Healthy": "o",    # Circle
    "Acne_NL": "^",    # Triangle
    "Acne_L": "D",     # Star
}

# Define the order for the legend
severity_order = ["absent Healthy", "absent Acne_NL", "low Acne_NL", "low Acne_L", "moderate Acne_L", "high Acne_L"]

# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom'),
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom')
]

# First, filter the metadata to get the samples to be removed
samples_to_remove = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']['SampleID'].tolist()

# Function to filter the biom table based on the samples to keep
def filter_biom_table(biom_table, samples_to_remove):
    # Get all sample IDs in the biom table
    all_samples = biom_table.ids(axis='sample')
    
    # Filter the samples, excluding the ones to be removed
    samples_to_keep = [sample for sample in all_samples if sample not in samples_to_remove]
    
    # Subset the biom table to only include the samples to keep
    filtered_biom_table = biom_table.filter(samples_to_keep, axis='sample', inplace=False)
    
    return filtered_biom_table, all_samples, samples_to_keep

# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table
    biom_tbl = biom.load_table(biom_file)
    
    # Filter the biom table to remove unwanted samples
    biom_tbl_filtered, original_samples, filtered_samples = filter_biom_table(biom_tbl, samples_to_remove)
    
    # Check the samples that were removed
    dropped_samples = set(original_samples) - set(filtered_samples)
    
    print(f"Number of samples in original table: {len(original_samples)}")
    print(f"Number of samples after filtering: {len(filtered_samples)}")
    print(f"Samples that were removed: {dropped_samples}")
    
    # Verify if the dropped samples match the expected samples
    if set(samples_to_remove) == dropped_samples:
        print(f"Correct samples were dropped: {len(dropped_samples)} samples removed.")
    else:
        print("There is a mismatch in the samples dropped. Please check!")
    
    # Use filtered biom table for RPCA
    np.seterr(divide='ignore')
    ordination, distance = rpca(biom_tbl_filtered)

    # Clear the figure to avoid overlapping plots
    plt.clf()
    
    # Extract and view sample ordinations from RPCA result
    spca_df = pd.DataFrame(ordination.samples)
    spca_df["severity_group"] = spca_df.index.map(metadata_df.set_index("SampleID")["severity_group"])
    spca_df["group"] = spca_df.index.map(metadata_df.set_index("SampleID")["group"])  # Add group mapping
    
    # Calculate permanova F-statistic
    pnova_res = permanova(distance, spca_df, "severity_group")
    p_value = pnova_res['p-value']
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors and shapes
    for severity_group in severity_order:
        group = spca_df[spca_df['severity_group'] == severity_group]
        for group_name, group_data in group.groupby('group'):
            sns.scatterplot(
                data=group_data,
                x="PC1",
                y="PC2",
                facecolor=severity_group_colors[severity_group],
                edgecolor=severity_group_colors[severity_group],
                alpha=0.8,
                linewidth=0.5,
                ax=ax,
                label=severity_group,
                marker=group_shape_mapping[group_name],  # Use shape mapping for each group
                s=10
            )
            
            # Calculate mean and covariance matrix for the group if more than 2 points
            points = group_data[["PC1", "PC2"]].values
            mean = np.mean(points, axis=0)
            
            if len(group_data) > 2:
                cov = np.cov(points, rowvar=False)
                draw_ellipse(mean, cov, ax, severity_group_colors[severity_group], scale_factor=2)

            # Add a subtle star (8-pointed) at the mean location for each group
            ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group], 
                       marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Add axis labels and include the explained variance percentages
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)

    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title based on the region being analyzed
    ax.text(0.05, 1.10, f'RPCA - 16S rRNA ({region_title})', fontsize=8, va='top', ha='left', transform=ax.transAxes)
    
    # Add the p-value from PERMANOVA
    ax.text(-0.18, 0.27, 'PERMANOVA', fontsize=7)
    ax.text(-0.18, 0.25, f'***p-val = {p_value:.3f}', fontsize=7)

    # Customize legend in the top-right corner
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7,
        loc='upper right'
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{region_title}_severity_group_RPCA.png", dpi=600)
    plt.show()


Processing dataset: 174951 - V4
Number of samples in original table: 217
Number of samples after filtering: 199
Samples that were removed: {'LAMI.RD306.D28.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD310.D7.C3', 'LAMI.RD304.D7.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD314.D14.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD314.D28.C3'}
There is a mismatch in the samples dropped. Please check!


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2662478306.py:65: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/437375351.py:115: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group],
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2662478306.py:65: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width,

Processing dataset: 174950 - V1-V3
Number of samples in original table: 236
Number of samples after filtering: 215
Samples that were removed: {'LAMI.RD310.D7.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD306.D28.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD308.D21.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD314.D28.C3', 'LAMI.RD306.D0.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD317.D21.C3', 'LAMI.RD319.D21.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD314.D14.C3'}
There is a mismatch in the samples dropped. Please check!


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2662478306.py:65: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/437375351.py:115: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group],
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2662478306.py:65: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width,

In [129]:
#single elipse
# Set the random seed for reproducibility
np.random.seed(123)

# Define the color palette for the groups
severity_group_colors = {
    "absent Healthy": "#006f00",
    "absent Acne_NL": "#9af09b",
    "low Acne_L": "#ff676c",
    "moderate Acne_L": "#ca0000", 
    "high Acne_L": "#960000"
}

# Define the shape for each group
group_shape_mapping = {
    "Healthy": "o",    # Circle
    "Acne_NL": "^",    # Triangle
    "Acne_L": "D",     # Star
}

# Define the order for the legend
severity_order = ["absent Healthy", "absent Acne_NL", "low Acne_NL", "low Acne_L", "moderate Acne_L", "high Acne_L"]

# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom'),
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom')
]

# First, filter the metadata to get the samples to be removed
samples_to_remove = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']['SampleID'].tolist()

# Function to filter the biom table based on the samples to keep
def filter_biom_table(biom_table, samples_to_remove):
    # Get all sample IDs in the biom table
    all_samples = biom_table.ids(axis='sample')
    
    # Filter the samples, excluding the ones to be removed
    samples_to_keep = [sample for sample in all_samples if sample not in samples_to_remove]
    
    # Subset the biom table to only include the samples to keep
    filtered_biom_table = biom_table.filter(samples_to_keep, axis='sample', inplace=False)
    
    return filtered_biom_table, all_samples, samples_to_keep

# Function to draw an ellipse
def draw_ellipse(mean, cov, ax, color, scale_factor=1):
    from matplotlib.patches import Ellipse
    import matplotlib.transforms as transforms
    import numpy as np

    # Get eigenvalues and eigenvectors of the covariance matrix
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]

    # Calculate the angle between the largest eigenvector and the x-axis
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))

    # Width and height are "full" lengths, so scale them by the factor
    width, height = 2 * np.sqrt(eigvals) * scale_factor

    # Draw the ellipse
    ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
    ax.add_patch(ellipse)

# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table
    biom_tbl = biom.load_table(biom_file)
    
    # Filter the biom table to remove unwanted samples
    biom_tbl_filtered, original_samples, filtered_samples = filter_biom_table(biom_tbl, samples_to_remove)
    
    # Check the samples that were removed
    dropped_samples = set(original_samples) - set(filtered_samples)
    
    print(f"Number of samples in original table: {len(original_samples)}")
    print(f"Number of samples after filtering: {len(filtered_samples)}")
    print(f"Samples that were removed: {dropped_samples}")
    
    # Verify if the dropped samples match the expected samples
    if set(samples_to_remove) == dropped_samples:
        print(f"Correct samples were dropped: {len(dropped_samples)} samples removed.")
    else:
        print("There is a mismatch in the samples dropped. Please check!")
    
    # Use filtered biom table for RPCA
    np.seterr(divide='ignore')
    ordination, distance = rpca(biom_tbl_filtered)

    # Clear the figure to avoid overlapping plots
    plt.clf()
    
    # Extract and view sample ordinations from RPCA result
    spca_df = pd.DataFrame(ordination.samples)
    spca_df["severity_group"] = spca_df.index.map(metadata_df.set_index("SampleID")["severity_group"])
    spca_df["group"] = spca_df.index.map(metadata_df.set_index("SampleID")["group"])  # Add group mapping
    
    # Calculate permanova F-statistic
    pnova_res = permanova(distance, spca_df, "severity_group")
    p_value = pnova_res['p-value']
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors and shapes
    for severity_group in severity_order:
        if severity_group in spca_df['severity_group'].values:
            group = spca_df[spca_df['severity_group'] == severity_group]
            for group_name, group_data in group.groupby('group'):
                sns.scatterplot(
                    data=group_data,
                    x="PC1",
                    y="PC2",
                    facecolor=severity_group_colors[severity_group],
                    edgecolor=severity_group_colors[severity_group],
                    alpha=0.8,
                    linewidth=0.5,
                    ax=ax,
                    label=severity_group,
                    marker=group_shape_mapping[group_name],  # Use shape mapping for each group
                    s=10
                )

    # Now plot a single ellipse around each of the main groups (Healthy, Acne_L, and Acne_NL)
    for group_name, group_data in spca_df.groupby('group'):
        points = group_data[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        
        if len(group_data) > 2:
            cov = np.cov(points, rowvar=False)
            # Define unique colors for the ellipses around the main groups
            ellipse_color = {
                "Healthy": "#006f00",    # Green for Healthy
                "Acne_NL": "#9af09b",    # Light green for Acne_NL
                "Acne_L": "#ff676c"      # Red for Acne_L
            }[group_name]

            draw_ellipse(mean, cov, ax, ellipse_color, scale_factor=2)

    # Add star markers for the mean location of each severity group (except dropped 'low Acne_NL')
    for severity_group in severity_order:
        if severity_group in spca_df['severity_group'].values:  # Add check to ensure severity group is present
            group = spca_df[spca_df['severity_group'] == severity_group]
            points = group[["PC1", "PC2"]].values
            mean = np.mean(points, axis=0)
            
            # Add a star marker at the mean location of each severity group
            ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group], 
                       marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Add axis labels and include the explained variance percentages
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)

    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title based on the region being analyzed
    ax.text(0.05, 1.10, f'RPCA - 16S rRNA ({region_title})', fontsize=8, va='top', ha='left', transform=ax.transAxes)
    
    # Add the p-value from PERMANOVA
    ax.text(-0.18, 0.27, 'PERMANOVA', fontsize=7)
    ax.text(-0.18, 0.25, f'***p-val = {p_value:.3f}', fontsize=7)

    # Customize legend in the top-right corner
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7,
        loc='upper right'
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{region_title}_severity_group_RPCA.png", dpi=600)
    plt.show()


Processing dataset: 174951 - V4
Number of samples in original table: 217
Number of samples after filtering: 199
Samples that were removed: {'LAMI.RD306.D28.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD310.D7.C3', 'LAMI.RD304.D7.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD314.D14.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD314.D28.C3'}
There is a mismatch in the samples dropped. Please check!


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3016173210.py:64: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3016173210.py:64: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3016173210.py:64: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, c

Processing dataset: 174950 - V1-V3
Number of samples in original table: 236
Number of samples after filtering: 215
Samples that were removed: {'LAMI.RD310.D7.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD306.D28.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD308.D21.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD314.D28.C3', 'LAMI.RD306.D0.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD317.D21.C3', 'LAMI.RD319.D21.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD314.D14.C3'}
There is a mismatch in the samples dropped. Please check!


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3016173210.py:64: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3016173210.py:64: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3016173210.py:64: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, c